# TP 6 — Réparer trois graphiques

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/curat/data-science/notebooks/TP6_Reparer_trois_graphiques.ipynb)

**Data Sciences, visualisation et Big Data** · M2 Télédétection et SIG ·
Séance 6 · KOUADIO Kouao Laurent

Pour chaque graphique « avant » :
1. écrivez en **une phrase** le message que le lecteur doit voir ;
2. listez les **erreurs** (liste de contrôle du cours) ;
3. tracez une version **« après »**.

Les données viennent des séances 2, 4 et 5. Durée : environ 75
minutes.

In [ ]:
# Google Colab : installe les paquets du cours (environ 1 minute).
# Dans l'environnement conda du cours (environment.yml), rien à faire.
import sys
if "google.colab" in sys.modules:
    %pip install -q pystac-client

## 0. Imports et outils

In [ ]:
import io

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pystac_client import Client


def deuteranopie(rgb):
    """Simule une deutéranopie (Machado et al., 2009).

    rgb : tableau (..., 3) de valeurs sRGB entre 0 et 1.
    """
    m = np.array([[0.367322, 0.860646, -0.227968],
                  [0.280085, 0.672501, 0.047413],
                  [-0.011820, 0.042940, 0.968881]])
    lin = np.where(rgb <= 0.04045, rgb / 12.92,
                   ((rgb + 0.055) / 1.055) ** 2.4)
    out = np.clip(lin @ m.T, 0, 1)
    return np.where(out <= 0.0031308, 12.92 * out,
                    1.055 * out ** (1 / 2.4) - 0.055)


def voir_comme_daltonien(fig):
    """Affiche une figure Matplotlib telle qu'un deutéranope la voit."""
    tampon = io.BytesIO()  # marche même après plt.show()
    fig.savefig(tampon, format="png", dpi=80)
    tampon.seek(0)
    img = plt.imread(tampon)[..., :3]
    plt.figure(figsize=(8, 4))
    plt.imshow(deuteranopie(img))
    plt.axis("off")
    plt.show()

## 1. Graphique A : l'occupation du sol à Yamoussoukro

Parts des classes ESA WorldCover 2021 dans la zone de la séance 5 (%).

In [ ]:
parts = pd.Series({"Arbres": 38.26, "Arbustes": 32.2, "Herbacées": 24.39, "Cultures": 0.23, "Bâti": 4.23, "Sol nu": 0.33, "Eau": 0.16, "Zones humides": 0.21})

# AVANT
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(parts.values, labels=parts.index, autopct="%.0f%%",
       explode=[0.1] * len(parts), shadow=True)
ax.set_title("Occupation du sol")
plt.show()

**Votre message (une phrase) :** ...

**Erreurs repérées :** ...

In [ ]:
# APRÈS : à vous de jouer
# barres horizontales triées, valeurs au bout des barres, unité, titre-message et source

## 2. Graphique B : le NDVI au Nord et au Sud

NDVI médian mensuel en 2025 (séance 4). Les mois absents n'ont aucune
image claire.

In [ ]:
ndvi = {"Nord (savane, Korhogo)": {"1": 0.332, "2": 0.546, "3": 0.419, "4": 0.352, "5": 0.307, "6": 0.387, "8": 0.329, "9": 0.649, "10": 0.595, "11": 0.52, "12": 0.4}, "Sud (forêt, Taï)": {"1": 0.681, "2": 0.712, "3": 0.808, "4": 0.816, "5": 0.742, "12": 0.755}}
ndvi = {k: {int(m): v for m, v in c.items()} for k, c in ndvi.items()}

# AVANT
fig, ax = plt.subplots(figsize=(8, 4))
for k, (nom, c) in enumerate(ndvi.items()):
    ax.plot(list(c), list(c.values()), "o-", label=nom,
            color=plt.cm.jet(0.15 + 0.7 * k))
ax.set_ylim(0.3, 0.85)
ax.grid(True)
ax.legend(loc="lower right")
plt.show()
voir_comme_daltonien(fig)

**Votre message :** ...

**Erreurs repérées :** (pensez aux mois sans données, à l'axe Y, aux
couleurs et à la légende)

In [ ]:
MOIS = ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]

In [ ]:
# APRÈS : à vous de jouer
# NaN pour les mois absents, axe Y de 0 à 1, deux couleurs distinctes, étiquettes directes ; vérifiez avec voir_comme_daltonien(fig)

## 3. Graphique C : les nuages sur Abidjan

Couverture nuageuse de chaque scène Sentinel-2 de 2025 sur la tuile
30NUL (séance 2). Une seule requête au catalogue STAC.

In [ ]:
cat = Client.open("https://earth-search.aws.element84.com/v1")
its = cat.search(collections=["sentinel-2-l2a"],
                 bbox=[-4.10, 5.25, -3.85, 5.45],
                 datetime="2025").item_collection()
df = pd.DataFrame([{"mois": int(i.properties["datetime"][5:7]),
                    "nuages": i.properties["eo:cloud_cover"]}
                   for i in its
                   if i.properties["grid:code"] == "MGRS-30NUL"])
print(len(df), "scènes")

# AVANT
moy = df.groupby("mois")["nuages"].mean()
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(moy.index, moy.values, color=plt.cm.jet(moy.values / 100))
ax.set_ylim(40, 100)
plt.show()

**Votre message :** ...

**Erreurs repérées :** ...

In [ ]:
# APRÈS : à vous de jouer
# boîtes par mois + un point par scène, axe de 0 à 100, une seule couleur, ligne du seuil de 20 %

## 4. Bonus : une version interactive avec Plotly

On sauvegarde un fichier HTML que l'on peut ouvrir dans un navigateur
ou joindre à un rapport.

In [ ]:
import plotly.express as px

long = pd.DataFrame([{"site": nom, "mois": m, "ndvi": v}
                     for nom, c in ndvi.items() for m, v in c.items()])
fig = px.line(long, x="mois", y="ndvi", color="site", markers=True,
              range_y=[0, 1], labels={"ndvi": "NDVI médian"},
              color_discrete_sequence=["#C9A227", "#2D6A4F"])
fig.write_html("ndvi_interactif.html")
fig

## À rendre

Le notebook exécuté, avec pour chaque graphique : le message en une
phrase, la liste des erreurs et votre version « après ». Comparez
votre version à celle proposée et justifiez vos choix.